# Aula 2 - Física Computacional: Modelagem e Simulação

**Nexus - Núcleo de Estudos em Física Computacional**

Na Aula 1, vimos a base de programação científica em Python: variáveis, funções, condicionais, laços, listas, NumPy, Matplotlib e Pandas.

Nesta aula, vamos usar esses conceitos para construir modelos físicos simples. A ideia é aprender o caminho entre uma equação física e uma simulação computacional.

Esta aula prepara vocês para o projeto trainee do plano inclinado, mas não resolve o projeto. Vamos praticar com problemas parecidos:

- movimento vertical com aceleração constante;
- lançamento vertical;
- bloco puxado em uma superfície horizontal;
- movimento com atrito cinético;
- comparação entre solução analítica e solução numérica;
- organização de resultados em gráficos e tabelas.

Ao final desta aula, você deve conseguir:

- identificar as variáveis de um modelo físico;
- escrever uma função para calcular aceleração;
- atualizar posição e velocidade passo a passo;
- interpretar o papel do passo de tempo `dt`;
- comparar simulação numérica com resultado analítico;
- organizar uma simulação em funções reutilizáveis.

## 1. Importando as bibliotecas

Vamos usar as mesmas ferramentas da Aula 1:

- `NumPy` para cálculos numéricos;
- `Matplotlib` para gráficos;
- `Pandas` para tabelas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## 2. O que significa modelar um problema físico?

Modelar é transformar uma situação física real em uma versão matemática e computacional.

Um bom modelo deixa claro:

- qual sistema estamos estudando;
- quais grandezas serão calculadas;
- quais forças ou interações são importantes;
- quais hipóteses foram feitas;
- quais unidades estão sendo usadas.

Por exemplo, em um movimento vertical próximo da superfície da Terra, podemos começar com o modelo mais simples:

- desprezar resistência do ar;
- usar gravidade constante;
- considerar apenas uma dimensão;
- usar o eixo vertical `y`.

Nesse caso, a aceleração é constante:

$$a = -g$$

O sinal negativo aparece porque vamos escolher o sentido positivo para cima.

## 3. Solução analítica: queda e lançamento vertical

Quando a aceleração é constante, podemos calcular posição e velocidade diretamente:

$$y(t) = y_0 + v_0t + \frac{1}{2}at^2$$

$$v(t) = v_0 + at$$

Essas fórmulas são chamadas de solução analítica. Elas servem como referência para testar uma simulação numérica.

In [ ]:
def posicao_analitica(t, y0, v0, a):
    return y0 + v0 * t + 0.5 * a * t ** 2


def velocidade_analitica(t, v0, a):
    return v0 + a * t


g = 9.81
tempo = np.linspace(0, 3, 100)
y = posicao_analitica(tempo, y0=20.0, v0=0.0, a=-g)
v = velocidade_analitica(tempo, v0=0.0, a=-g)

plt.figure(figsize=(7, 4))
plt.plot(tempo, y, label="posição")
plt.plot(tempo, v, label="velocidade")
plt.xlabel("tempo (s)")
plt.ylabel("valor")
plt.title("Queda vertical sem resistência do ar")
plt.legend()
plt.grid(True)
plt.show()

### Exercício 1

Um objeto é lançado verticalmente para cima com:

- `y0 = 0 m`;
- `v0 = 15 m/s`;
- `a = -9.81 m/s^2`;
- tempo de `0` a `3 s`.

Use as funções acima para calcular posição e velocidade. Depois, gere um gráfico com as duas curvas.

In [ ]:
# Resolva aqui

## 4. Simulação numérica passo a passo

Em muitos problemas, não usamos diretamente uma fórmula pronta para `x(t)` ou `y(t)`. Em vez disso, atualizamos o estado do sistema em pequenos intervalos de tempo.

Se conhecemos posição, velocidade, aceleração e um intervalo pequeno `dt`, podemos fazer:

$$v_{novo} = v_{antigo} + a\,dt$$

$$y_{novo} = y_{antigo} + v_{novo}\,dt$$

Essa versão é conhecida como Euler-Cromer. Ela é parecida com o método de Euler e é bastante usada em problemas introdutórios de mecânica computacional.

In [ ]:
y = 20.0
v = 0.0
a = -9.81
dt = 0.1

for passo in range(10):
    v = v + a * dt
    y = y + v * dt
    print(f"passo = {passo}, y = {y:.3f} m, v = {v:.3f} m/s")

## 5. Guardando a evolução temporal

Para fazer gráficos, precisamos guardar os valores calculados a cada passo.

Vamos guardar:

- tempo;
- posição;
- velocidade.

In [ ]:
y = 20.0
v = 0.0
a = -9.81
dt = 0.01
tempo_total = 2.0

n_passos = int(tempo_total / dt)

tempos = np.zeros(n_passos + 1)
posicoes = np.zeros(n_passos + 1)
velocidades = np.zeros(n_passos + 1)

posicoes[0] = y
velocidades[0] = v

for i in range(n_passos):
    tempos[i + 1] = tempos[i] + dt
    velocidades[i + 1] = velocidades[i] + a * dt
    posicoes[i + 1] = posicoes[i] + velocidades[i + 1] * dt

plt.figure(figsize=(7, 4))
plt.plot(tempos, posicoes)
plt.xlabel("tempo (s)")
plt.ylabel("posição vertical (m)")
plt.title("Queda vertical simulada")
plt.grid(True)
plt.show()

### Exercício 2

Modifique a simulação anterior para representar um lançamento vertical com:

- `y = 0 m`;
- `v = 15 m/s`;
- `a = -9.81 m/s^2`;
- `dt = 0.01 s`;
- `tempo_total = 3 s`.

Gere dois gráficos: posição por tempo e velocidade por tempo.

In [ ]:
# Resolva aqui

## 6. Transformando a simulação em função

Uma simulação fica mais útil quando transformamos o código em função.

Assim conseguimos testar vários casos mudando apenas os parâmetros.

In [ ]:
def simular_movimento_1d(x0, v0, a, dt, tempo_total):
    n_passos = int(tempo_total / dt)

    tempos = np.zeros(n_passos + 1)
    posicoes = np.zeros(n_passos + 1)
    velocidades = np.zeros(n_passos + 1)

    posicoes[0] = x0
    velocidades[0] = v0

    for i in range(n_passos):
        tempos[i + 1] = tempos[i] + dt
        velocidades[i + 1] = velocidades[i] + a * dt
        posicoes[i + 1] = posicoes[i] + velocidades[i + 1] * dt

    return tempos, posicoes, velocidades


tempos, posicoes, velocidades = simular_movimento_1d(
    x0=0.0,
    v0=15.0,
    a=-9.81,
    dt=0.01,
    tempo_total=3.0
)

plt.figure(figsize=(7, 4))
plt.plot(tempos, posicoes)
plt.xlabel("tempo (s)")
plt.ylabel("posição vertical (m)")
plt.title("Lançamento vertical simulado com função")
plt.grid(True)
plt.show()

## 7. Comparando solução numérica e analítica

Sempre que possível, devemos testar a simulação em um caso conhecido.

Aqui, como a aceleração é constante, temos uma solução analítica. Vamos comparar com a solução numérica.

In [ ]:
y0 = 0.0
v0 = 15.0
a = -9.81
dt = 0.1
tempo_total = 3.0

tempos, y_num, v_num = simular_movimento_1d(y0, v0, a, dt, tempo_total)
y_exato = posicao_analitica(tempos, y0, v0, a)

erro = y_num - y_exato

plt.figure(figsize=(7, 4))
plt.plot(tempos, y_exato, label="analítico")
plt.plot(tempos, y_num, "--", label="numérico")
plt.xlabel("tempo (s)")
plt.ylabel("posição vertical (m)")
plt.title("Solução analítica x solução numérica")
plt.legend()
plt.grid(True)
plt.show()

print("erro máximo:", np.max(np.abs(erro)))

### Exercício 3

Repita a comparação para:

- `dt = 0.5`;
- `dt = 0.1`;
- `dt = 0.01`.

Calcule o erro máximo em cada caso.

Pergunta: o que acontece com o erro quando diminuímos `dt`?

In [ ]:
# Resolva aqui

## 8. Forças e segunda lei de Newton

Até agora usamos a aceleração diretamente. Em muitos problemas físicos, primeiro calculamos a força resultante.

A segunda lei de Newton diz:

$$F_{resultante} = ma$$

Logo:

$$a = \frac{F_{resultante}}{m}$$

Vamos estudar um bloco em uma superfície horizontal sendo puxado por uma força constante.

Hipóteses iniciais:

- movimento em uma dimensão;
- superfície sem atrito;
- força aplicada constante;
- massa constante.

In [ ]:
def aceleracao_por_forca(forca_resultante, massa):
    return forca_resultante / massa


massa = 2.0
forca = 10.0

a = aceleracao_por_forca(forca, massa)
print(f"aceleração = {a:.2f} m/s^2")

## 9. Bloco puxado sem atrito

Agora vamos simular o bloco puxado por uma força constante em superfície horizontal sem atrito.

In [ ]:
massa = 2.0
forca = 10.0
a = aceleracao_por_forca(forca, massa)


tempos, posicoes, velocidades = simular_movimento_1d(
    x0=0.0,
    v0=0.0,
    a=a,
    dt=0.01,
    tempo_total=4.0
)

plt.figure(figsize=(7, 4))
plt.plot(tempos, posicoes, label="posição")
plt.plot(tempos, velocidades, label="velocidade")
plt.xlabel("tempo (s)")
plt.ylabel("valor")
plt.title("Bloco puxado sem atrito")
plt.legend()
plt.grid(True)
plt.show()

### Exercício 4

Simule o bloco sem atrito para três forças aplicadas:

- `F = 2 N`;
- `F = 5 N`;
- `F = 10 N`.

Use:

- `massa = 2 kg`;
- `x0 = 0 m`;
- `v0 = 0 m/s`;
- `dt = 0.01 s`;
- `tempo_total = 4 s`.

Gere um único gráfico com a posição em função do tempo para as três forças.

In [ ]:
# Resolva aqui

## 10. Incluindo atrito cinético em superfície horizontal

Em uma superfície horizontal, a força normal é:

$$N = mg$$

A força de atrito cinético tem módulo:

$$F_{atrito} = \mu_k N$$

Então:

$$F_{atrito} = \mu_k mg$$

Se a força aplicada puxa o bloco para a direita e o bloco se move para a direita, o atrito aponta para a esquerda.

A força resultante fica:

$$F_{resultante} = F_{aplicada} - F_{atrito}$$

E a aceleração:

$$a = \frac{F_{aplicada} - \mu_kmg}{m}$$

In [ ]:
def aceleracao_bloco_horizontal(forca_aplicada, massa, mu_k, g=9.81):
    forca_atrito = mu_k * massa * g
    forca_resultante = forca_aplicada - forca_atrito
    return forca_resultante / massa


massa = 2.0
forca = 10.0
mu_k = 0.20

a = aceleracao_bloco_horizontal(forca, massa, mu_k)
print(f"aceleração com atrito = {a:.2f} m/s^2")

## 11. Comparando com e sem atrito

Vamos comparar dois modelos:

- bloco puxado sem atrito;
- bloco puxado com atrito cinético.

In [ ]:
massa = 2.0
forca = 10.0
mu_k = 0.20

a_sem_atrito = aceleracao_por_forca(forca, massa)
a_com_atrito = aceleracao_bloco_horizontal(forca, massa, mu_k)

tempos, x_sem, v_sem = simular_movimento_1d(0.0, 0.0, a_sem_atrito, 0.01, 4.0)
tempos, x_com, v_com = simular_movimento_1d(0.0, 0.0, a_com_atrito, 0.01, 4.0)

plt.figure(figsize=(7, 4))
plt.plot(tempos, x_sem, label="sem atrito")
plt.plot(tempos, x_com, label="com atrito")
plt.xlabel("tempo (s)")
plt.ylabel("posição (m)")
plt.title("Efeito do atrito no movimento")
plt.legend()
plt.grid(True)
plt.show()

### Exercício 5

Para um bloco de `2 kg` puxado por uma força de `10 N`, simule o movimento com:

- `mu_k = 0.0`;
- `mu_k = 0.1`;
- `mu_k = 0.2`;
- `mu_k = 0.4`.

Gere um gráfico da posição em função do tempo para todos os valores de atrito.

Pergunta: como o aumento de `mu_k` altera a aceleração?

In [ ]:
# Resolva aqui

## 12. Atrito estático: quando o bloco começa a se mover?

Antes de escorregar, o bloco pode ficar parado por causa do atrito estático.

Em uma superfície horizontal, o maior atrito estático possível é:

$$F_{estatico,max} = \mu_s mg$$

O bloco começa a se mover se:

$$F_{aplicada} > F_{estatico,max}$$

Caso contrário, ele permanece parado.

In [ ]:
def bloco_horizontal_anda(forca_aplicada, massa, mu_s, g=9.81):
    forca_estatica_max = mu_s * massa * g
    return forca_aplicada > forca_estatica_max


massa = 2.0
mu_s = 0.50
forca = 8.0

if bloco_horizontal_anda(forca, massa, mu_s):
    print("O bloco começa a se mover.")
else:
    print("O bloco permanece parado.")

### Exercício 6

Para um bloco de `2 kg` com `mu_s = 0.50`, teste forças aplicadas de `0 N` a `20 N`.

Descubra aproximadamente a menor força que faz o bloco começar a se mover.

Dica: use `np.linspace`, um laço `for` e a função `bloco_horizontal_anda`.

In [ ]:
# Resolva aqui

## 13. Um modelo com decisão física

Agora vamos montar uma função um pouco mais realista para o bloco horizontal.

A função vai:

1. verificar se a força aplicada vence o atrito estático;
2. se não vencer, retornar posição e velocidade constantes;
3. se vencer, calcular a aceleração com atrito cinético;
4. simular o movimento.

Esse tipo de estrutura é importante para projetos maiores: o código precisa tomar decisões com base na Física.

In [ ]:
def simular_bloco_horizontal(forca_aplicada, massa, mu_s, mu_k, x0, v0, dt, tempo_total, g=9.81):
    n_passos = int(tempo_total / dt)

    if not bloco_horizontal_anda(forca_aplicada, massa, mu_s, g):
        tempos = np.linspace(0, tempo_total, n_passos + 1)
        posicoes = np.full(n_passos + 1, x0)
        velocidades = np.full(n_passos + 1, 0.0)
        aceleracao = 0.0
        return tempos, posicoes, velocidades, aceleracao

    aceleracao = aceleracao_bloco_horizontal(forca_aplicada, massa, mu_k, g)
    tempos, posicoes, velocidades = simular_movimento_1d(x0, v0, aceleracao, dt, tempo_total)

    return tempos, posicoes, velocidades, aceleracao


tempos, posicoes, velocidades, aceleracao = simular_bloco_horizontal(
    forca_aplicada=12.0,
    massa=2.0,
    mu_s=0.50,
    mu_k=0.30,
    x0=0.0,
    v0=0.0,
    dt=0.01,
    tempo_total=4.0
)

print(f"aceleração usada no modelo: {aceleracao:.2f} m/s^2")

plt.figure(figsize=(7, 4))
plt.plot(tempos, posicoes)
plt.xlabel("tempo (s)")
plt.ylabel("posição (m)")
plt.title("Bloco horizontal com decisão de atrito")
plt.grid(True)
plt.show()

## 14. Organizando resultados com Pandas

Tabelas ajudam a comparar simulações e registrar resultados importantes.

In [ ]:
dados = pd.DataFrame({
    "tempo (s)": tempos,
    "posição (m)": posicoes,
    "velocidade (m/s)": velocidades
})

dados.head()

In [ ]:
print("posição final:", dados["posição (m)"].iloc[-1])
print("velocidade final:", dados["velocidade (m/s)"].iloc[-1])
print("velocidade média:", dados["velocidade (m/s)"].mean())

### Exercício 7

Crie uma tabela comparando posição final, velocidade final e aceleração para quatro forças aplicadas:

- `5 N`;
- `10 N`;
- `15 N`;
- `20 N`.

Use:

- `massa = 2 kg`;
- `mu_s = 0.50`;
- `mu_k = 0.30`;
- `x0 = 0 m`;
- `v0 = 0 m/s`;
- `dt = 0.01 s`;
- `tempo_total = 4 s`.

A tabela deve ter as colunas:

- `força aplicada (N)`;
- `aceleração (m/s^2)`;
- `posição final (m)`;
- `velocidade final (m/s)`.

In [ ]:
# Resolva aqui

## 15. Miniaplicação: lançamento vertical com condição de parada

Em uma simulação simples de queda ou lançamento, podemos parar quando o objeto toca o chão.

Isso exige uma condição dentro do laço:

```python
if y <= 0:
    break
```

Vamos simular uma bola lançada para cima a partir do chão e parar a simulação quando ela voltar ao solo.

In [ ]:
def simular_lancamento_ate_o_chao(y0, v0, dt, g=9.81):
    y = y0
    v = v0
    t = 0.0

    tempos = [t]
    posicoes = [y]
    velocidades = [v]

    while True:
        v = v - g * dt
        y = y + v * dt
        t = t + dt

        tempos.append(t)
        posicoes.append(y)
        velocidades.append(v)

        if y <= 0 and t > 0:
            break

    return np.array(tempos), np.array(posicoes), np.array(velocidades)


tempos, posicoes, velocidades = simular_lancamento_ate_o_chao(
    y0=0.0,
    v0=20.0,
    dt=0.01
)

plt.figure(figsize=(7, 4))
plt.plot(tempos, posicoes)
plt.xlabel("tempo (s)")
plt.ylabel("altura (m)")
plt.title("Lan?amento vertical at? retornar ao solo")
plt.grid(True)
plt.show()

print(f"tempo de voo aproximado: {tempos[-1]:.2f} s")
print(f"altura m?xima aproximada: {np.max(posicoes):.2f} m")

### Exercício 8

Use a função `simular_lancamento_ate_o_chao` para comparar três velocidades iniciais:

- `10 m/s`;
- `20 m/s`;
- `30 m/s`.

Gere um gráfico da altura em função do tempo para os três casos.

Depois, monte uma tabela com:

- velocidade inicial;
- tempo de voo;
- altura máxima.

In [ ]:
# Resolva aqui

## 16. O que esta aula prepara para o projeto trainee?

O projeto do plano inclinado vai exigir exatamente as ideias praticadas aqui:

- decompor forças;
- transformar força resultante em aceleração;
- decidir quando existe ou não movimento;
- atualizar posição e velocidade no tempo;
- testar casos simples;
- comparar cenários com e sem atrito;
- gerar gráficos e tabelas;
- escrever funções reutilizáveis.

A diferença é que, no projeto, vocês vão aplicar essas ideias à geometria do plano inclinado.

O ponto central é: antes de programar, definam bem o modelo físico.

## 17. Mini-projeto da aula

Crie uma simulação completa para o bloco horizontal puxado por uma força constante.

Sua simulação deve receber:

- `forca_aplicada`;
- `massa`;
- `mu_s`;
- `mu_k`;
- `x0`;
- `v0`;
- `dt`;
- `tempo_total`.

Ela deve produzir:

- gráfico de posição por tempo;
- gráfico de velocidade por tempo;
- tabela com tempo, posição e velocidade;
- posição final;
- velocidade final;
- uma mensagem dizendo se o bloco ficou parado ou entrou em movimento.

Use como caso inicial:

- `forca_aplicada = 14 N`;
- `massa = 3 kg`;
- `mu_s = 0.35`;
- `mu_k = 0.25`;
- `x0 = 0 m`;
- `v0 = 0 m/s`;
- `dt = 0.01 s`;
- `tempo_total = 5 s`.

In [ ]:
# Mini-projeto: desenvolva sua solução aqui

## 18. Resumo da aula

Nesta aula, vimos:

- como transformar um problema físico em código;
- solução analítica para aceleração constante;
- simulação numérica passo a passo;
- método de Euler-Cromer;
- influência do passo de tempo `dt`;
- segunda lei de Newton em código;
- bloco horizontal com e sem atrito;
- atrito estático como condição de início do movimento;
- organização de resultados com Pandas;
- estrutura de um mini-projeto físico computacional.

Essas ideias são a base para construir, depois, o modelo do plano inclinado com e sem atrito.